In [ ]:
%pip install -q deepinv==0.5.0 scikit-image==0.25.2 matplotlib==3.10.0 pandas==2.2.3


In [ ]:

import torch
import deepinv as dinv

print("PyTorch version:", torch.__version__)
print("DeepInverse version:", dinv.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Setup completed successfully!")


In [ ]:

import torch
import deepinv as dinv
import numpy as np
import matplotlib.pyplot as plt

from skimage import data, transform

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load sample RGB image
original = data.astronaut()

# Resize to 128x128 for faster experiments
original = transform.resize(
    original,
    (128, 128),
    anti_aliasing=True
)

# Convert to PyTorch tensor [Batch, Channels, Height, Width]
x = torch.from_numpy(
    original.astype(np.float32)
).permute(2, 0, 1).unsqueeze(0).to(device)

print("Device:", device)
print("Image shape:", x.shape)

plt.figure(figsize=(5, 5))
plt.imshow(original)
plt.title("Original Clean Image")
plt.axis("off")
plt.show()


In [ ]:

# Define Gaussian blur kernel
blur_kernel = dinv.physics.functional.gaussian_blur(
    sigma=(2.0, 2.0)
)

# Create blur physics operator
physics = dinv.physics.BlurFFT(
    x.shape[1:],
    filter=blur_kernel,
    device=device
)

# Add Gaussian noise
physics.noise_model = dinv.physics.GaussianNoise(
    sigma=0.03
)

# Generate degraded image
y = physics(x)

print("Blur and noise successfully added!")
print("Degraded image shape:", y.shape)


In [ ]:

# Convert tensors into images
clean_img = x[0].permute(1, 2, 0).detach().cpu().numpy()
noisy_img = y[0].permute(1, 2, 0).detach().cpu().numpy()

# Display comparison
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].imshow(np.clip(clean_img, 0, 1))
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(np.clip(noisy_img, 0, 1))
axes[1].set_title("Gaussian Blur + Noise")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("step2_degradation.png", dpi=200)
plt.show()

print("Comparison saved successfully!")


In [ ]:

from skimage.metrics import peak_signal_noise_ratio

psnr = peak_signal_noise_ratio(
    clean_img,
    noisy_img,
    data_range=1.0
)

print(f"Degraded image PSNR: {psnr:.2f} dB")


In [ ]:

import torch
import deepinv as dinv

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load pretrained DRUNet denoiser
denoiser = dinv.models.DRUNet(
    pretrained="download",
    device=device
)

denoiser.eval()

print("DRUNet loaded successfully!")
print("Running on:", device)


In [ ]:

# Create Deep Plug-and-Play restoration model
model = dinv.optim.DPIR(
    sigma=0.03,
    denoiser=denoiser,
    device=device
)

model.eval()

# Restore the blurred + noisy image
with torch.no_grad():
    x_restored = model(y, physics)

print("Image restoration completed!")
print("Restored shape:", x_restored.shape)


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio

# Convert PyTorch tensors to NumPy images
def to_image(t):
    return t[0].detach().cpu().permute(1, 2, 0).numpy()

original_img = to_image(x)
degraded_img = to_image(y)
restored_img = to_image(x_restored)

# Calculate PSNR
before_psnr = peak_signal_noise_ratio(
    original_img,
    degraded_img,
    data_range=1.0
)

after_psnr = peak_signal_noise_ratio(
    original_img,
    np.clip(restored_img, 0, 1),
    data_range=1.0
)

# Display comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

images = [
    original_img,
    degraded_img,
    restored_img
]

titles = [
    "Original Image",
    f"Blur + Noise\nPSNR: {before_psnr:.2f} dB",
    f"DPIR Restored\nPSNR: {after_psnr:.2f} dB"
]

for ax, image, title in zip(axes, images, titles):
    ax.imshow(np.clip(image, 0, 1))
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.savefig(
    "deepinverse_restoration_comparison.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

print(f"Before restoration: {before_psnr:.2f} dB")
print(f"After restoration:  {after_psnr:.2f} dB")
print(f"PSNR improvement:   {after_psnr-before_psnr:+.2f} dB")


In [ ]:
import time
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# Helper function
def to_image(t):
    return t[0].detach().cpu().permute(1, 2, 0).numpy()

# Time the restoration
start_time = time.time()

with torch.no_grad():
    x_restored = model(y, physics)

end_time = time.time()
runtime_sec = end_time - start_time

# Convert to numpy
original_img = to_image(x)
degraded_img = to_image(y)
restored_img = np.clip(to_image(x_restored), 0, 1)

# Metrics
before_psnr = peak_signal_noise_ratio(original_img, degraded_img, data_range=1.0)
after_psnr = peak_signal_noise_ratio(original_img, restored_img, data_range=1.0)

before_ssim = structural_similarity(original_img, degraded_img, channel_axis=2, data_range=1.0)
after_ssim = structural_similarity(original_img, restored_img, channel_axis=2, data_range=1.0)

print(f"Before restoration PSNR: {before_psnr:.2f} dB")
print(f"After restoration PSNR:  {after_psnr:.2f} dB")
print(f"PSNR improvement:        {after_psnr - before_psnr:+.2f} dB")
print()
print(f"Before restoration SSIM: {before_ssim:.4f}")
print(f"After restoration SSIM:  {after_ssim:.4f}")
print(f"SSIM improvement:        {after_ssim - before_ssim:+.4f}")
print()
print(f"Runtime: {runtime_sec:.2f} seconds")

In [ ]:
import pandas as pd

results_df = pd.DataFrame([
    {
        "image_name": "astronaut",
        "blur_sigma": 2.0,
        "noise_sigma": 0.03,
        "psnr_before": before_psnr,
        "psnr_after": after_psnr,
        "psnr_improvement": after_psnr - before_psnr,
        "ssim_before": before_ssim,
        "ssim_after": after_ssim,
        "ssim_improvement": after_ssim - before_ssim,
        "runtime_sec": runtime_sec
    }
])

results_df.to_csv("results_metrics.csv", index=False)
results_df

In [ ]:

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from skimage import data, transform
from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity
)

# Reuse the trained-model weights and physics
# from Step 3. No need to download them again.
model.eval()
torch.manual_seed(42)

# Four RGB test images
test_images = {
    "astronaut": data.astronaut,
    "chelsea": data.chelsea,
    "coffee": data.coffee,
    "rocket": data.rocket
}

results = []
visual_results = {}

def tensor_to_image(tensor):
    return (
        tensor[0]
        .detach()
        .cpu()
        .permute(1, 2, 0)
        .numpy()
    )

def synchronize_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

for name, load_image in test_images.items():

    print(f"Processing: {name}")

    # Load and normalize image
    img = transform.resize(
        load_image(),
        (128, 128),
        anti_aliasing=True
    ).astype(np.float32)

    clean = (
        torch.from_numpy(img)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .to(device)
    )

    # Apply same blur + noise physics
    with torch.no_grad():
        degraded = physics(clean)

    # Warm up the model on first image
    if len(results) == 0:
        with torch.no_grad():
            _ = model(degraded, physics)
        synchronize_gpu()

    # Measure actual restoration time
    synchronize_gpu()
    start = time.perf_counter()

    with torch.no_grad():
        restored = model(degraded, physics)

    synchronize_gpu()
    runtime = time.perf_counter() - start

    # Convert and clip to valid image range
    original_np = np.clip(
        tensor_to_image(clean), 0, 1
    )
    degraded_np = np.clip(
        tensor_to_image(degraded), 0, 1
    )
    restored_np = np.clip(
        tensor_to_image(restored), 0, 1
    )

    # Compute PSNR
    psnr_before = peak_signal_noise_ratio(
        original_np, degraded_np, data_range=1.0
    )
    psnr_after = peak_signal_noise_ratio(
        original_np, restored_np, data_range=1.0
    )

    # Compute SSIM
    ssim_before = structural_similarity(
        original_np, degraded_np,
        channel_axis=2, data_range=1.0
    )
    ssim_after = structural_similarity(
        original_np, restored_np,
        channel_axis=2, data_range=1.0
    )

    results.append({
        "image": name,
        "psnr_before": psnr_before,
        "psnr_after": psnr_after,
        "psnr_gain": psnr_after - psnr_before,
        "ssim_before": ssim_before,
        "ssim_after": ssim_after,
        "ssim_gain": ssim_after - ssim_before,
        "runtime_sec": runtime
    })

    visual_results[name] = (
        original_np,
        degraded_np,
        restored_np
    )

    print(
        f"PSNR: {psnr_before:.2f} -> "
        f"{psnr_after:.2f} dB"
    )
    print(f"Runtime: {runtime:.3f} sec\n")

print("All four experiments completed!")


In [ ]:

# Create benchmark DataFrame
df = pd.DataFrame(results)

# Save complete numerical results
df.to_csv(
    "multi_image_restoration_results.csv",
    index=False
)

# Display results
display(df.round(4))

print("\n===== BENCHMARK SUMMARY =====")

print(
    f"Mean PSNR improvement: "
    f"{df['psnr_gain'].mean():+.2f} dB"
)

print(
    f"Mean SSIM improvement: "
    f"{df['ssim_gain'].mean():+.4f}"
)

print(
    f"Mean restoration runtime: "
    f"{df['runtime_sec'].mean():.3f} sec"
)

print(
    f"Images with improved PSNR: "
    f"{(df['psnr_gain'] > 0).sum()}/{len(df)}"
)


In [ ]:

fig, axes = plt.subplots(
    4, 3, figsize=(12, 14)
)

for row, (name, images) in enumerate(
    visual_results.items()
):
    original, degraded, restored = images

    scores = df[df["image"] == name].iloc[0]

    titles = [
        f"{name.upper()} — Original",
        f"Blur + Noise\nPSNR: {scores['psnr_before']:.2f} dB",
        f"DPIR Restored\nPSNR: {scores['psnr_after']:.2f} dB"
    ]

    for col, (image, title) in enumerate(
        zip(images, titles)
    ):
        axes[row, col].imshow(image)
        axes[row, col].set_title(
            title, fontsize=10
        )
        axes[row, col].axis("off")

plt.suptitle(
    "Multi-Image Plug-and-Play Restoration Benchmark",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.97])

plt.savefig(
    "multi_image_restoration_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

import torch
import deepinv as dinv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from skimage import data, transform
from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Reuse pretrained DRUNet from Step 3
denoiser.eval()

# Controlled degradation conditions
conditions = [
    ("E1", 1.0, 0.01),
    ("E2", 2.0, 0.03),
    ("E3", 3.0, 0.05)
]

# Same 4 RGB images
images = {
    "astronaut": data.astronaut,
    "chelsea": data.chelsea,
    "coffee": data.coffee,
    "rocket": data.rocket
}

experiment_results = []
example_outputs = {}

def to_numpy(t):
    return (
        t[0].detach().cpu()
        .permute(1, 2, 0).numpy()
    )

def sync():
    if device == "cuda":
        torch.cuda.synchronize()

for exp_id, blur_sigma, noise_sigma in conditions:

    print(f"\n===== {exp_id} =====")
    print(f"Blur: {blur_sigma}, Noise: {noise_sigma}")

    # Create Gaussian blur kernel
    kernel = dinv.physics.functional.gaussian_blur(
        sigma=(blur_sigma, blur_sigma),
        device=device
    )

    # Create degradation physics
    physics_case = dinv.physics.BlurFFT(
        img_size=(3, 128, 128),
        filter=kernel,
        device=device,
        noise_model=dinv.physics.GaussianNoise(
            sigma=noise_sigma
        )
    )

    # Match DPIR noise parameter to measurement noise
    restorer = dinv.optim.DPIR(
        sigma=noise_sigma,
        denoiser=denoiser,
        device=device
    )
    restorer.eval()

    for image_index, (name, loader) in enumerate(images.items()):

        # Load original image
        img = transform.resize(
            loader(),
            (128, 128),
            anti_aliasing=True
        ).astype(np.float32)

        clean = (
            torch.from_numpy(img)
            .permute(2, 0, 1)
            .unsqueeze(0)
            .to(device)
        )

        # Repeatable noise pattern for each image
        seed = 2026 + image_index
        torch.manual_seed(seed)

        with torch.no_grad():
            degraded = physics_case(clean)

        # Warm-up (excluded from runtime)
        with torch.no_grad():
            _ = restorer(degraded, physics_case)

        sync()
        start = time.perf_counter()

        with torch.no_grad():
            restored = restorer(degraded, physics_case)

        sync()
        runtime = time.perf_counter() - start

        original_np = np.clip(to_numpy(clean), 0, 1)
        degraded_np = np.clip(to_numpy(degraded), 0, 1)
        restored_np = np.clip(to_numpy(restored), 0, 1)

        # PSNR
        before_psnr = peak_signal_noise_ratio(
            original_np, degraded_np, data_range=1.0
        )
        after_psnr = peak_signal_noise_ratio(
            original_np, restored_np, data_range=1.0
        )

        # SSIM
        before_ssim = structural_similarity(
            original_np, degraded_np,
            channel_axis=2, data_range=1.0
        )
        after_ssim = structural_similarity(
            original_np, restored_np,
            channel_axis=2, data_range=1.0
        )

        experiment_results.append({
            "experiment": exp_id,
            "image": name,
            "blur_sigma": blur_sigma,
            "noise_sigma": noise_sigma,
            "seed": seed,
            "psnr_before": before_psnr,
            "psnr_after": after_psnr,
            "psnr_gain": after_psnr - before_psnr,
            "ssim_before": before_ssim,
            "ssim_after": after_ssim,
            "ssim_gain": after_ssim - before_ssim,
            "runtime_sec": runtime
        })

        # Save one image example per condition
        if name == "astronaut":
            example_outputs[exp_id] = (
                original_np,
                degraded_np,
                restored_np
            )

        print(
            f"{name}: PSNR "
            f"{before_psnr:.2f} -> {after_psnr:.2f} dB"
        )

print("\nAll 12 experiments completed!")


In [ ]:

# Full results
df_robustness = pd.DataFrame(experiment_results)

df_robustness.to_csv(
    "step6_robustness_full_results.csv",
    index=False
)

display(df_robustness.round(4))

# Calculate mean performance per experiment
summary = df_robustness.groupby(
    ["experiment", "blur_sigma", "noise_sigma"],
    as_index=False
).agg(
    mean_psnr_before=("psnr_before", "mean"),
    mean_psnr_after=("psnr_after", "mean"),
    mean_psnr_gain=("psnr_gain", "mean"),
    mean_ssim_before=("ssim_before", "mean"),
    mean_ssim_after=("ssim_after", "mean"),
    mean_ssim_gain=("ssim_gain", "mean"),
    mean_runtime=("runtime_sec", "mean")
)

print("\n===== ROBUSTNESS SUMMARY =====")
display(summary.round(4))

summary.to_csv(
    "step6_robustness_summary.csv",
    index=False
)


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

labels = summary["experiment"].tolist()
xpos = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    xpos - width/2,
    summary["mean_psnr_before"],
    width,
    label="Degraded image"
)

ax.bar(
    xpos + width/2,
    summary["mean_psnr_after"],
    width,
    label="DPIR restored"
)

ax.set_ylabel("Mean PSNR (dB)")
ax.set_xlabel("Degradation experiment")
ax.set_title(
    "DPIR Restoration under Different Degradation Levels"
)
ax.set_xticks(xpos)
ax.set_xticklabels(labels)
ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(
    "step6_psnr_robustness.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:

fig, axes = plt.subplots(3, 3, figsize=(12, 11))

for row, exp_id in enumerate(["E1", "E2", "E3"]):
    original, degraded, restored = example_outputs[exp_id]

    images_to_show = [original, degraded, restored]
    titles = [
        f"{exp_id} - Original",
        f"{exp_id} - Degraded",
        f"{exp_id} - DPIR Restored"
    ]

    for col in range(3):
        axes[row, col].imshow(images_to_show[col])
        axes[row, col].set_title(titles[col])
        axes[row, col].axis("off")

plt.tight_layout()
plt.savefig(
    "step6_visual_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:

import torch
import deepinv as dinv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from skimage import data, transform
from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Same degradation condition as Step 6, E2
BLUR_SIGMA = 2.0
NOISE_SIGMA = 0.03

# Build the forward operator
kernel = dinv.physics.functional.gaussian_blur(
    sigma=(BLUR_SIGMA, BLUR_SIGMA),
    device=device
)

physics_e2 = dinv.physics.BlurFFT(
    img_size=(3, 128, 128),
    filter=kernel,
    device=device,
    noise_model=dinv.physics.GaussianNoise(
        sigma=NOISE_SIGMA
    )
)

# Method A: Classical Total Variation + PGD
# Fixed exploratory settings from the TV approach
def create_tv_model():
    return dinv.optim.PGD(
        data_fidelity=dinv.optim.L2(),
        prior=dinv.optim.TVPrior(n_it_max=20),
        lambda_reg=0.01,
        stepsize=1.0,
        max_iter=100,
        early_stop=False
    )

# Method B: Deep Plug-and-Play DPIR
denoiser.eval()

dpir_model = dinv.optim.DPIR(
    sigma=NOISE_SIGMA,
    denoiser=denoiser,
    device=device
)
dpir_model.eval()

# Test images
test_images = {
    "astronaut": data.astronaut,
    "chelsea": data.chelsea,
    "coffee": data.coffee,
    "rocket": data.rocket
}

step7_rows = []
step7_visuals = {}

def to_np(t):
    return (
        t[0].detach().cpu()
        .permute(1, 2, 0).numpy()
    )

def sync_gpu():
    if device == "cuda":
        torch.cuda.synchronize()

def calculate_metrics(reference, prediction):
    return (
        peak_signal_noise_ratio(
            reference, prediction, data_range=1.0
        ),
        structural_similarity(
            reference, prediction,
            channel_axis=2, data_range=1.0
        )
    )

for i, (name, loader) in enumerate(test_images.items()):

    print(f"\nProcessing: {name}")

    img = transform.resize(
        loader(),
        (128, 128),
        anti_aliasing=True
    ).astype(np.float32)

    clean = (
        torch.from_numpy(img)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .to(device)
    )

    # Same measurement for both algorithms
    torch.manual_seed(2026 + i)

    with torch.no_grad():
        degraded = physics_e2(clean)

    # Classical TV model, fresh per image
    tv_model = create_tv_model()
    tv_model.eval()

    sync_gpu()
    t0 = time.perf_counter()

    with torch.no_grad():
        tv_output = tv_model(degraded, physics_e2)

    sync_gpu()
    tv_time = time.perf_counter() - t0

    # DPIR restoration
    sync_gpu()
    t0 = time.perf_counter()

    with torch.no_grad():
        dpir_output = dpir_model(
            degraded, physics_e2
        )

    sync_gpu()
    dpir_time = time.perf_counter() - t0

    # Convert to RGB images
    original = np.clip(to_np(clean), 0, 1)
    observed = np.clip(to_np(degraded), 0, 1)
    tv_img = np.clip(to_np(tv_output), 0, 1)
    dpir_img = np.clip(to_np(dpir_output), 0, 1)

    # Evaluation
    p0, s0 = calculate_metrics(original, observed)
    ptv, stv = calculate_metrics(original, tv_img)
    pdpir, sdpir = calculate_metrics(
        original, dpir_img
    )

    step7_rows.append({
        "image": name,
        "psnr_degraded": p0,
        "psnr_tv": ptv,
        "psnr_dpir": pdpir,
        "ssim_degraded": s0,
        "ssim_tv": stv,
        "ssim_dpir": sdpir,
        "tv_runtime_sec": tv_time,
        "dpir_runtime_sec": dpir_time
    })

    step7_visuals[name] = (
        original, observed, tv_img, dpir_img
    )

    print(f"Degraded PSNR: {p0:.2f} dB")
    print(f"TV-PGD PSNR:   {ptv:.2f} dB")
    print(f"DPIR PSNR:     {pdpir:.2f} dB")
    print(f"TV time:       {tv_time:.3f} sec")
    print(f"DPIR time:     {dpir_time:.3f} sec")

print("\nStep 7 completed!")


In [ ]:

df_step7 = pd.DataFrame(step7_rows)

df_step7["dpir_minus_tv_psnr"] = (
    df_step7["psnr_dpir"] - df_step7["psnr_tv"]
)

df_step7.to_csv(
    "step7_tv_vs_dpir_results.csv",
    index=False
)

display(df_step7.round(4))

print("\n===== METHOD COMPARISON =====")

print(
    "Mean TV-PGD PSNR:",
    round(df_step7["psnr_tv"].mean(), 2)
)

print(
    "Mean DPIR PSNR:",
    round(df_step7["psnr_dpir"].mean(), 2)
)

print(
    "Mean DPIR - TV PSNR:",
    round(df_step7["dpir_minus_tv_psnr"].mean(), 2)
)

print(
    "Mean TV runtime:",
    round(df_step7["tv_runtime_sec"].mean(), 3)
)

print(
    "Mean DPIR runtime:",
    round(df_step7["dpir_runtime_sec"].mean(), 3)
)


In [ ]:

fig, axes = plt.subplots(4, 4, figsize=(15, 14))

for row, (name, imgs) in enumerate(step7_visuals.items()):

    scores = df_step7[
        df_step7["image"] == name
    ].iloc[0]

    titles = [
        f"{name} - Original",
        f"Degraded: {scores['psnr_degraded']:.2f} dB",
        f"TV-PGD: {scores['psnr_tv']:.2f} dB",
        f"DPIR: {scores['psnr_dpir']:.2f} dB"
    ]

    for col in range(4):
        axes[row, col].imshow(imgs[col])
        axes[row, col].set_title(titles[col], fontsize=10)
        axes[row, col].axis("off")

fig.suptitle(
    "Classical TV-PGD vs Deep Plug-and-Play Restoration",
    fontsize=15
)

plt.tight_layout(rect=[0, 0, 1, 0.97])

plt.savefig(
    "step7_method_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage import data, transform

# Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(2026)

# Load the same astronaut image
img = transform.resize(
    data.astronaut(),
    (128, 128),
    anti_aliasing=True
).astype(np.float32)

clean = (
    torch.from_numpy(img)
    .permute(2, 0, 1)
    .unsqueeze(0)
    .to(device)
)

# Same E2 blur and noise from Step 7
with torch.no_grad():
    degraded = physics_e2(clean)

# Method A: TV-PGD
tv_convergence_model = create_tv_model()
tv_convergence_model.eval()

# Method B: DPIR
dpir_model.eval()

# Compute iteration-level metrics
with torch.no_grad():

    tv_output, tv_metrics = tv_convergence_model(
        degraded,
        physics_e2,
        x_gt=clean,
        compute_metrics=True
    )

    dpir_output, dpir_metrics = dpir_model(
        degraded,
        physics_e2,
        x_gt=clean,
        compute_metrics=True
    )

print("Convergence analysis completed!")
print("TV-PGD iterations:",
      len(tv_metrics["residual"][0]))

print("DPIR iterations:",
      len(dpir_metrics["residual"][0]))

print("Final TV-PGD logged PSNR:",
      round(tv_metrics["psnr"][0][-1], 2))

print("Final DPIR logged PSNR:",
      round(dpir_metrics["psnr"][0][-1], 2))


In [ ]:

def metrics_to_dataframe(method_name, metrics):

    psnr = np.asarray(
        metrics["psnr"][0], dtype=float
    )

    residual = np.asarray(
        metrics["residual"][0], dtype=float
    )

    # Initial PSNR + one value per iteration
    assert len(psnr) == len(residual) + 1, (
        "Unexpected metric history length"
    )

    # Residual is undefined at iteration zero
    residual_with_initial = np.concatenate(
        ([np.nan], residual)
    )

    return pd.DataFrame({
        "method": method_name,
        "iteration": np.arange(len(psnr)),
        "psnr_db": psnr,
        "relative_residual": residual_with_initial
    })

tv_history = metrics_to_dataframe(
    "TV-PGD", tv_metrics
)

dpir_history = metrics_to_dataframe(
    "DPIR", dpir_metrics
)

convergence_df = pd.concat(
    [tv_history, dpir_history],
    ignore_index=True
)

convergence_df.to_csv(
    "step8_convergence_metrics.csv",
    index=False
)

print("Convergence metrics saved!")
display(convergence_df.head(15).round(6))


In [ ]:

plt.figure(figsize=(9, 5))

for method, group in convergence_df.groupby("method"):

    plt.plot(
        group["iteration"],
        group["psnr_db"],
        marker="o",
        markersize=3,
        label=method
    )

plt.xlabel("Iteration")
plt.ylabel("PSNR (dB)")
plt.title("Reconstruction Quality Across Iterations")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    "step8_psnr_vs_iterations.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

plt.figure(figsize=(9, 5))

for method, group in convergence_df.groupby("method"):

    valid = group.dropna(
        subset=["relative_residual"]
    )

    plt.semilogy(
        valid["iteration"],
        np.maximum(
            valid["relative_residual"].to_numpy(),
            1e-12
        ),
        marker="o",
        markersize=3,
        label=method
    )

plt.xlabel("Iteration")
plt.ylabel("Relative Update Residual (log scale)")

plt.title(
    "Numerical Iteration Behavior: TV-PGD vs DPIR"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    "step8_convergence_residual.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

print("===== STEP 8 SUMMARY =====")

for method, group in convergence_df.groupby("method"):

    final_psnr = group["psnr_db"].iloc[-1]

    residual_values = group[
        "relative_residual"
    ].dropna()

    final_residual = residual_values.iloc[-1]

    iterations = len(residual_values)

    best_psnr = group["psnr_db"].max()
    best_iteration = group.loc[
        group["psnr_db"].idxmax(),
        "iteration"
    ]

    print(f"\nMethod: {method}")
    print(f"Iterations: {iterations}")
    print(f"Final PSNR: {final_psnr:.2f} dB")
    print(f"Best PSNR: {best_psnr:.2f} dB")
    print(f"Best iteration: {best_iteration}")
    print(f"Final residual: {final_residual:.6e}")

print("\nAnalysis completed successfully!")


In [ ]:

import torch
import torch.nn.functional as F
import deepinv as dinv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from skimage import data, transform
from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Step 9: Two-level coarse-to-fine experiment
# Fine resolution: 128x128
# Coarse resolution: 64x64

FINE_SIZE = 128
COARSE_SIZE = 64
NOISE_SIGMA = 0.03
BLUR_SIGMA = 2.0

# Approximate coarse model
# Averaging 2x2 independent noise pixels
# reduces noise standard deviation by 2.
COARSE_NOISE = NOISE_SIGMA / 2
COARSE_BLUR = BLUR_SIGMA / 2

coarse_kernel = dinv.physics.functional.gaussian_blur(
    sigma=(COARSE_BLUR, COARSE_BLUR),
    device=device
)

coarse_physics = dinv.physics.BlurFFT(
    img_size=(3, COARSE_SIZE, COARSE_SIZE),
    filter=coarse_kernel,
    device=device,
    noise_model=dinv.physics.GaussianNoise(
        sigma=COARSE_NOISE
    )
)

# Coarse-scale DPIR with same pretrained denoiser
coarse_model = dinv.optim.DPIR(
    sigma=COARSE_NOISE,
    denoiser=denoiser,
    device=device
)

coarse_model.eval()
dpir_model.eval()

print("Two-level framework initialized!")
print("Fine resolution:", FINE_SIZE)
print("Coarse resolution:", COARSE_SIZE)
print("GPU:", device)


In [ ]:

test_images = {
    "astronaut": data.astronaut,
    "chelsea": data.chelsea,
    "coffee": data.coffee,
    "rocket": data.rocket
}

multilevel_results = []
multilevel_visuals = {}

def sync_gpu():
    if device == "cuda":
        torch.cuda.synchronize()

def run_timed(fn):
    sync_gpu()
    start = time.perf_counter()

    with torch.no_grad():
        result = fn()

    sync_gpu()
    return result, time.perf_counter() - start

def to_image(t):
    return np.clip(
        t[0].detach().cpu()
        .permute(1, 2, 0).numpy(),
        0, 1
    )

def evaluate(ref, output):
    psnr = peak_signal_noise_ratio(
        ref, output, data_range=1.0
    )
    ssim = structural_similarity(
        ref, output,
        channel_axis=2,
        data_range=1.0
    )
    return psnr, ssim

for i, (name, loader) in enumerate(test_images.items()):

    print(f"\nProcessing: {name}")

    # Original image
    img = transform.resize(
        loader(),
        (128, 128),
        anti_aliasing=True
    ).astype(np.float32)

    clean = (
        torch.from_numpy(img)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .to(device)
    )

    # Generate one high-resolution observation
    torch.manual_seed(2026 + i)

    with torch.no_grad():
        y_full = physics_e2(clean)

    # Coarse observation obtained ONLY from y_full
    y_coarse = F.avg_pool2d(
        y_full, kernel_size=2
    )

    # Warm up models on first image
    if i == 0:
        with torch.no_grad():
            _ = dpir_model(y_full, physics_e2)
            _ = coarse_model(
                y_coarse, coarse_physics
            )
        sync_gpu()

    # Method A: Standard single-scale DPIR
    direct, direct_time = run_timed(
        lambda: dpir_model(
            y_full, physics_e2
        )
    )

    # Method B: Coarse reconstruction
    coarse, coarse_time = run_timed(
        lambda: coarse_model(
            y_coarse, coarse_physics
        )
    )

    # Upsample coarse reconstruction
    coarse_up = F.interpolate(
        coarse,
        size=(128, 128),
        mode="bilinear",
        align_corners=False
    ).clamp(0, 1)

    # Fine DPIR initialized by coarse reconstruction
    two_level, fine_time = run_timed(
        lambda: dpir_model(
            y_full,
            physics_e2,
            init=coarse_up
        )
    )

    total_two_level_time = (
        coarse_time + fine_time
    )

    # Convert to images
    ref = to_image(clean)
    observed = to_image(y_full)
    direct_img = to_image(direct)
    coarse_img = to_image(coarse_up)
    two_level_img = to_image(two_level)

    # Quality metrics
    direct_psnr, direct_ssim = evaluate(
        ref, direct_img
    )

    coarse_psnr, coarse_ssim = evaluate(
        ref, coarse_img
    )

    multi_psnr, multi_ssim = evaluate(
        ref, two_level_img
    )

    multilevel_results.append({
        "image": name,
        "psnr_direct": direct_psnr,
        "psnr_coarse_init": coarse_psnr,
        "psnr_two_level": multi_psnr,
        "psnr_difference": multi_psnr - direct_psnr,
        "ssim_direct": direct_ssim,
        "ssim_two_level": multi_ssim,
        "ssim_difference": multi_ssim - direct_ssim,
        "direct_runtime": direct_time,
        "coarse_runtime": coarse_time,
        "fine_runtime": fine_time,
        "two_level_runtime": total_two_level_time
    })

    multilevel_visuals[name] = (
        ref,
        observed,
        direct_img,
        coarse_img,
        two_level_img
    )

    print(f"Direct DPIR:  {direct_psnr:.2f} dB")
    print(f"Coarse init:  {coarse_psnr:.2f} dB")
    print(f"Two-level:    {multi_psnr:.2f} dB")
    print(
        f"PSNR change:  "
        f"{multi_psnr - direct_psnr:+.2f} dB"
    )
    print(
        f"Direct time:  {direct_time:.3f} sec"
    )
    print(
        f"Two-level:    "
        f"{total_two_level_time:.3f} sec"
    )

print("\nAll multilevel experiments completed!")


In [ ]:

df_multilevel = pd.DataFrame(multilevel_results)

display(df_multilevel.round(4))

df_multilevel.to_csv(
    "step9_multilevel_results.csv",
    index=False
)

print("\n===== MULTILEVEL SUMMARY =====")

print(
    f"Mean direct PSNR: "
    f"{df_multilevel['psnr_direct'].mean():.2f} dB"
)

print(
    f"Mean two-level PSNR: "
    f"{df_multilevel['psnr_two_level'].mean():.2f} dB"
)

print(
    f"Mean PSNR difference: "
    f"{df_multilevel['psnr_difference'].mean():+.3f} dB"
)

print(
    f"Mean SSIM difference: "
    f"{df_multilevel['ssim_difference'].mean():+.4f}"
)

print(
    f"Mean direct runtime: "
    f"{df_multilevel['direct_runtime'].mean():.3f} sec"
)

print(
    f"Mean two-level runtime: "
    f"{df_multilevel['two_level_runtime'].mean():.3f} sec"
)

print(
    "Images improved:",
    (df_multilevel["psnr_difference"] > 0).sum(),
    "/", len(df_multilevel)
)


In [ ]:

fig, axes = plt.subplots(
    4, 5, figsize=(18, 14)
)

for row, (name, images) in enumerate(
    multilevel_visuals.items()
):

    scores = df_multilevel[
        df_multilevel["image"] == name
    ].iloc[0]

    titles = [
        "Original",
        "Degraded",
        f"Direct DPIR\n{scores['psnr_direct']:.2f} dB",
        f"Coarse Initialization\n{scores['psnr_coarse_init']:.2f} dB",
        f"Two-Level DPIR\n{scores['psnr_two_level']:.2f} dB"
    ]

    for col, (image, title) in enumerate(
        zip(images, titles)
    ):
        axes[row, col].imshow(image)
        axes[row, col].set_title(
            title, fontsize=10
        )
        axes[row, col].axis("off")

    axes[row, 0].set_ylabel(
        name.upper(), fontsize=11
    )

fig.suptitle(
    "Single-Scale vs Two-Level Plug-and-Play Restoration",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.97])

plt.savefig(
    "step9_multilevel_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from skimage import data, transform
from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity
)

# Check previous models
required = [
    "physics_e2", "dpir_model",
    "coarse_model", "coarse_physics"
]

for name in required:
    assert name in globals(), f"Missing: {name}"

device = "cuda" if torch.cuda.is_available() else "cpu"

dpir_model.eval()
coarse_model.eval()

images = {
    "astronaut": data.astronaut,
    "chelsea": data.chelsea,
    "coffee": data.coffee,
    "rocket": data.rocket
}

REPEATS = 5
SIZE = 128
rows = []

def sync():
    if device == "cuda":
        torch.cuda.synchronize()

def measure(fn):
    sync()
    start = time.perf_counter()

    with torch.no_grad():
        output = fn()

    sync()
    return output, time.perf_counter() - start

def direct_reconstruction(y):
    return dpir_model(y, physics_e2)

def observation_init_reconstruction(y):
    # Coarse observation without learned restoration
    y_small = F.avg_pool2d(y, kernel_size=2)

    initial = F.interpolate(
        y_small,
        size=y.shape[-2:],
        mode="bilinear",
        align_corners=False
    ).clamp(0, 1)

    return dpir_model(
        y, physics_e2, init=initial
    )

def coarse_pnp_reconstruction(y):
    # Coarse DPIR stage
    y_small = F.avg_pool2d(y, kernel_size=2)

    coarse_result = coarse_model(
        y_small, coarse_physics
    )

    initial = F.interpolate(
        coarse_result,
        size=y.shape[-2:],
        mode="bilinear",
        align_corners=False
    ).clamp(0, 1)

    # Fine DPIR stage
    return dpir_model(
        y, physics_e2, init=initial
    )

methods = {
    "Direct DPIR": direct_reconstruction,
    "Observation Init": observation_init_reconstruction,
    "Coarse PnP Init": coarse_pnp_reconstruction
}

def to_np(t):
    return np.clip(
        t[0].detach().cpu()
        .permute(1, 2, 0).numpy(),
        0, 1
    )

# Reproducible evaluation
for image_index, (name, loader) in enumerate(images.items()):

    img = transform.resize(
        loader(),
        (SIZE, SIZE),
        anti_aliasing=True
    ).astype(np.float32)

    x = (
        torch.from_numpy(img)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .to(device)
    )

    original = to_np(x)

    for repeat in range(REPEATS):

        seed = 2026 + image_index + 100 * repeat
        torch.manual_seed(seed)

        # Generate measurement ONCE
        with torch.no_grad():
            y = physics_e2(x)

        # Same y for all 3 methods
        for method_name, method_fn in methods.items():

            # First-call warmup
            if image_index == 0 and repeat == 0:
                with torch.no_grad():
                    _ = method_fn(y)
                sync()

            restored, runtime = measure(
                lambda: method_fn(y)
            )

            output = to_np(restored)

            psnr = peak_signal_noise_ratio(
                original, output, data_range=1.0
            )

            ssim = structural_similarity(
                original, output,
                channel_axis=2,
                data_range=1.0
            )

            rows.append({
                "image": name,
                "repeat": repeat + 1,
                "seed": seed,
                "method": method_name,
                "psnr": psnr,
                "ssim": ssim,
                "runtime_sec": runtime
            })

        print(
            f"Completed: {name} | "
            f"Repeat {repeat + 1}/{REPEATS}"
        )

df10 = pd.DataFrame(rows)

assert len(df10) == 60

df10.to_csv(
    "step10_all_experiments.csv",
    index=False
)

print("\nAll 60 reconstruction runs completed!")
print("Rows:", len(df10))

display(df10.head(9).round(4))


In [ ]:

# Pair methods by exactly the same image and seed
wide = df10.pivot(
    index=["image", "seed"],
    columns="method",
    values=["psnr", "ssim", "runtime_sec"]
)

paired = pd.DataFrame(index=wide.index)

paired["direct_psnr"] = wide[
    ("psnr", "Direct DPIR")
]

paired["observation_gain"] = (
    wide[("psnr", "Observation Init")]
    - wide[("psnr", "Direct DPIR")]
)

paired["coarse_pnp_gain"] = (
    wide[("psnr", "Coarse PnP Init")]
    - wide[("psnr", "Direct DPIR")]
)

paired["coarse_ssim_gain"] = (
    wide[("ssim", "Coarse PnP Init")]
    - wide[("ssim", "Direct DPIR")]
)

paired = paired.reset_index()

paired.to_csv(
    "step10_paired_comparison.csv",
    index=False
)

# Per-image summary across five noise seeds
image_summary = paired.groupby(
    "image", as_index=False
).agg(
    observation_gain_mean=("observation_gain", "mean"),
    coarse_gain_mean=("coarse_pnp_gain", "mean"),
    coarse_gain_sd=("coarse_pnp_gain", "std"),
    coarse_ssim_gain_mean=("coarse_ssim_gain", "mean")
)

print("===== PER-IMAGE ABLATION SUMMARY =====")
display(image_summary.round(4))

print("\n===== OVERALL RESULTS =====")

print(
    "Mean coarse PnP PSNR gain:",
    round(paired["coarse_pnp_gain"].mean(), 4),
    "dB"
)

print(
    "SD of paired PSNR gains:",
    round(paired["coarse_pnp_gain"].std(), 4),
    "dB"
)

print(
    "Positive PSNR comparisons:",
    int((paired["coarse_pnp_gain"] > 0).sum()),
    "/ 20"
)

print(
    "Mean observation-init gain:",
    round(paired["observation_gain"].mean(), 4),
    "dB"
)

print("\n===== RUNTIME SUMMARY =====")

runtime_summary = df10.groupby(
    "method", as_index=False
).agg(
    mean_runtime=("runtime_sec", "mean"),
    std_runtime=("runtime_sec", "std")
)

display(runtime_summary.round(4))

image_summary.to_csv(
    "step10_image_summary.csv",
    index=False
)


In [ ]:

plot_df = image_summary.set_index("image")[
    ["observation_gain_mean", "coarse_gain_mean"]
]

ax = plot_df.plot(
    kind="bar",
    figsize=(10, 5),
    width=0.7
)

ax.axhline(0, linewidth=1)

ax.set_title(
    "Initialization Ablation: Mean PSNR Change vs Direct DPIR"
)

ax.set_xlabel("Test Image")
ax.set_ylabel("Mean Paired PSNR Difference (dB)")

ax.legend([
    "Observation Initialization",
    "Coarse PnP Initialization"
])

ax.grid(axis="y", alpha=0.25)

plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    "step10_initialization_ablation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

import pandas as pd
import numpy as np

# Validate completed Step 10 experiments
assert "df10" in globals(), "Run Step 10 first."

required_columns = {
    "image", "seed", "method",
    "psnr", "ssim", "runtime_sec"
}
assert required_columns.issubset(df10.columns)
assert len(df10) == 60, "Expected 60 runs."

counts = df10.groupby(
    ["image", "seed", "method"]
).size()

assert counts.eq(1).all(), "Duplicate experiments found."

# Same image and seed for all three methods
wide = df10.pivot(
    index=["image", "seed"],
    columns="method",
    values=["psnr", "ssim"]
)

paired = pd.DataFrame(index=wide.index)

paired["direct_psnr"] = wide[
    ("psnr", "Direct DPIR")
]

paired["observation_gain"] = (
    wide[("psnr", "Observation Init")]
    - wide[("psnr", "Direct DPIR")]
)

paired["coarse_gain"] = (
    wide[("psnr", "Coarse PnP Init")]
    - wide[("psnr", "Direct DPIR")]
)

paired["coarse_vs_observation"] = (
    wide[("psnr", "Coarse PnP Init")]
    - wide[("psnr", "Observation Init")]
)

paired["coarse_ssim_gain"] = (
    wide[("ssim", "Coarse PnP Init")]
    - wide[("ssim", "Direct DPIR")]
)

paired = paired.reset_index()

# Method-level summary
method_summary = df10.groupby(
    "method", as_index=False
).agg(
    mean_psnr=("psnr", "mean"),
    mean_ssim=("ssim", "mean"),
    mean_runtime=("runtime_sec", "mean")
)

# Image-level summary
image_summary11 = paired.groupby(
    "image", as_index=False
).agg(
    mean_observation_gain=("observation_gain", "mean"),
    mean_coarse_gain=("coarse_gain", "mean"),
    sd_coarse_gain=("coarse_gain", "std"),
    mean_coarse_vs_observation=(
        "coarse_vs_observation", "mean"
    )
)

print("===== VERIFIED METHOD SUMMARY =====")
display(method_summary.round(4))

print("\n===== PER-IMAGE SUMMARY =====")
display(image_summary11.round(4))

print("\n===== OVERALL ABLATION RESULTS =====")

print(
    "Mean coarse vs direct gain:",
    f"{paired['coarse_gain'].mean():+.4f} dB"
)

print(
    "Mean observation vs direct gain:",
    f"{paired['observation_gain'].mean():+.4f} dB"
)

print(
    "Mean coarse vs observation gain:",
    f"{paired['coarse_vs_observation'].mean():+.4f} dB"
)

print(
    "Positive coarse vs direct comparisons:",
    int((paired["coarse_gain"] > 0).sum()),
    "/ 20"
)

# Save all verified summaries
paired.to_csv(
    "step11_paired_results.csv", index=False
)

method_summary.to_csv(
    "step11_method_summary.csv", index=False
)

image_summary11.to_csv(
    "step11_image_summary.csv", index=False
)

print("\nValidation completed successfully!")
